In [1]:
import torch
from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np

d:\Projects\Customer Support AI Agent\csa\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Projects\Customer Support AI Agent\csa\Lib\site-packages\torch\compiler\__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


In [ ]:
import os 
import json
from Training.config import MODEL_NAME,MAX_LENGTH,BATCH_SIZE,LR,OUTPUT_DIR

In [3]:
EPOCHS = 10

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Load the Dataset

---

In [5]:

dataset = load_dataset("banking77")

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})

In [6]:
train_dataset = dataset["train"]
test_dataset = dataset["test"]

num_labels = len(set(train_dataset["label"]))
print(f"Number of labels: {num_labels}")

Number of labels: 77


### Creating Validation Split

In [7]:
train_dataset = train_dataset.train_test_split(test_size=0.1, seed=42)
val_dataset = train_dataset["test"]
train_dataset = train_dataset["train"]

## Map the Labels:

---

In [8]:
label_names = dataset["train"].features["label"].names
num_labels = len(label_names)

print(num_labels)
print(label_names)

77
['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire', 'card_acceptance', 'card_arrival', 'card_delivery_estimate', 'card_linking', 'card_not_working', 'card_payment_fee_charged', 'card_payment_not_recognised', 'card_payment_wrong_exchange_rate', 'card_swallowed', 'cash_withdrawal_charge', 'cash_withdrawal_not_recognised', 'change_pin', 'compromised_card', 'contactless_not_working', 'country_support', 'declined_card_payment', 'declined_cash_withdrawal', 'declined_transfer', 'direct_debit_payment_not_recognised', 'disposable_card_limits', 'edit_personal_details', 'exchange_charge', 'exchange_rate', 'exchange_via_app', 'extra_charge_on_statement', 'failed_transfer', 'fiat_currency_support', 'get_disposable_virtual_card', 'get_physical_card', 'getting_spare_card', 'getting_virtual_ca

In [9]:
id2label = {i: label_names[i] for i in range(num_labels)}
label2id = {label_names[i]: i for i in range(num_labels)}

# Save label mapping
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(f"{OUTPUT_DIR}/label_mapping.json", "w") as f:
    json.dump(id2label, f, indent=4)

## Load The Tokenizer

---

In [10]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

In [11]:
tokenizer

DistilBertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

### Tokenization Function

In [12]:
def tokenize_function(data):
    """
    Production-ready tokenization:
    - No static padding
    - Uses truncation
    - Efficient for GPU training
    """
    return tokenizer(
        data["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [13]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

### Adding Dynamic Padding for Optimization:

In [14]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Defining the Model

---

In [15]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

model.to(device)

Loading weights: 100%|██████████| 100/100 [00:00<?, ?it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

---

### Function to calculate different type of metrics:

In [44]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="weighted"),
        "recall": recall_score(labels, preds, average="weighted"),
        "f1": f1_score(labels, preds, average="weighted")
    }

## Training Arguments

---

In [17]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,

    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,

    metric_for_best_model="f1",
    greater_is_better=True,

    weight_decay=0.01,

    # GPU optimization
    fp16=True,

    logging_steps=100,
    save_total_limit=2,

    report_to="none"
)


In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


## Training the Transformer...

---

In [19]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,2.441510,1.896426,0.722278,0.716976,0.722278,0.688490
2,0.913719,0.791227,0.846154,0.852063,0.846154,0.834437
3,0.469174,0.472732,0.880120,0.881262,0.880120,0.875900
4,0.282852,0.374551,0.907093,0.913063,0.907093,0.905936
5,0.154709,0.349003,0.911089,0.918828,0.911089,0.910765
6,0.102703,0.330410,0.910090,0.915503,0.910090,0.910111
7,0.070116,0.351596,0.916084,0.921854,0.916084,0.915666
8,0.049513,0.335843,0.918082,0.922704,0.918082,0.917999
9,0.031737,0.349413,0.918082,0.922964,0.918082,0.917902
10,0.022930,0.351684,0.919081,0.923983,0.919081,0.918890


d:\Projects\Customer Support AI Agent\csa\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.09it/s]
d:\Projects\Customer Support AI Agent\csa\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.55it/s]
d:\Projects\Customer Support AI Agent\csa\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels w

TrainOutput(global_step=5630, training_loss=0.5877669751961744, metrics={'train_runtime': 176.853, 'train_samples_per_second': 509.01, 'train_steps_per_second': 31.834, 'total_flos': 911438189742048.0, 'train_loss': 0.5877669751961744, 'epoch': 10.0})

## Validation and Testing 

---

In [30]:
val_dataset = val_dataset.rename_column("label","labels")

In [32]:
val_dataset = val_dataset.remove_columns(["text"])

In [39]:
test_dataset = test_dataset.remove_columns(['text'])
test_dataset = test_dataset.rename_column("label","labels")

In [33]:
val_dataset

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 1001
})

In [40]:
test_dataset

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 3080
})

### Calculating the Performance Metrics Manually:

In [ ]:
# from torch.utils.data import DataLoader

# test_loader = DataLoader(test_dataset, batch_size=16, collate_fn=data_collator)

# model = trainer.model
# model.eval()

# all_preds = []
# all_labels = []

# with torch.no_grad():
#     for batch in test_loader:
#         batch = {k: v.to(device) for k, v in batch.items()}
        
#         outputs = model(**batch)
#         logits = outputs.logits
        
#         preds = torch.argmax(logits, dim=1)
        
#         all_preds.extend(preds.cpu().numpy())
#         all_labels.extend(batch["labels"].cpu().numpy())


# print({
#     "accuracy": accuracy_score(all_labels, all_preds),
#     "precision": precision_score(all_labels, all_preds, average="weighted"),
#     "recall": recall_score(all_labels, all_preds, average="weighted"),
#     "f1": f1_score(all_labels, all_preds, average="weighted")
# })


{'accuracy': 0.9292207792207792, 'precision': 0.9320441207213993, 'recall': 0.9292207792207792, 'f1': 0.9291645504228238}


## Calculating Performance Metrics for Validation and Test Datasets

In [46]:
val_pred = trainer.predict(val_dataset)

compute_metrics(val_pred)

{'accuracy': 0.919080919080919,
 'precision': 0.9239825806831942,
 'recall': 0.919080919080919,
 'f1': 0.9188901195444927}

In [45]:
pred = trainer.predict(test_dataset)

compute_metrics(pred)

{'accuracy': 0.9292207792207792,
 'precision': 0.9320441207213993,
 'recall': 0.9292207792207792,
 'f1': 0.9291645504228238}

## Saving the Model and Tokenizer

---

In [49]:
trainer.save_model("./intent_model")
tokenizer.save_pretrained("./intent_model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.17it/s]


('./intent_model\\tokenizer_config.json', './intent_model\\tokenizer.json')